# Tập 3.2 · Kalman dễ hiểu nhất 🔔

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap32/tap32_kalman.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 3 · ⏱ ~35 phút · Giáo trình: Principles of Robot Autonomy, Mục 12.2 (Thuật toán 1, Ví dụ 12.2.1)

Cảm biến nhiễu + gia tốc kế trôi = bộ lọc Kalman tốt hơn cả hai. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video.
2. **Tự viết** bước dự đoán, bước cập nhật và cả bộ lọc (3 bài), có ô chấm tự động.
3. Vẽ sai số, hệ số K và độ bất định bằng Kalman của bạn.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap32")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import kalman as goc               # mô phỏng trong video
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
p = np.array([r["p"] for r in goc.LOG]); z = np.array([r["z"] for r in goc.LOG])
mu = np.array([r["mu"] for r in goc.LOG]); dr = np.array([r["dr"] for r in goc.LOG])
rms = lambda e: float(np.sqrt(np.mean(e ** 2)))
print(f"Chỉ cảm biến: sai RMS {rms(z - p) * 100:.1f} cm · chỉ gia tốc kế: cuối lệch {abs(dr[-1] - p[-1]):.2f} m · Kalman: {rms(mu - p) * 100:.1f} cm")
print(f"K bước 1 = {goc.LOG[0]['K']:.3f}, cuối = {goc.LOG[-1]['K']:.3f}")

## 2. Bài tập

Dự đoán: $\bar\mu = A\mu + Bu$, $\bar\Sigma = A\Sigma A^T + Q$. Cập nhật: $S = C\bar\Sigma C^T + R$, $K = \bar\Sigma C^T S^{-1}$, $\mu = \bar\mu + K(z - C\bar\mu)$, $\Sigma = (I - KC)\bar\Sigma(I - KC)^T + KRK^T$.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `du_doan(mu, Sig, u)` | trả về `(μ̄, Σ̄)` |
| **2** | `cap_nhat(mu_b, Sig_b, z)` | trả về `(μ, Σ, K)` |
| **3** | `loc_kalman(ds_u, ds_z, mu0, Sig0)` | chạy cả chuỗi, trả về các μ |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np

DT = 0.1
SIGMA_A, SIGMA_Z = 0.3, 0.3
A = np.array([[1, DT], [0, 1]])
B = np.array([DT ** 2 / 2, DT])
C = np.array([[1.0, 0.0]])
Q = np.outer(B, B) * SIGMA_A ** 2
R = np.array([[SIGMA_Z ** 2]])


def du_doan(mu, Sig, u):
    """Bài 1 (dự đoán): đẩy niềm tin (μ, Σ) tới một bước.

        μ̄ = A·μ + B·u          Σ̄ = A·Σ·Aᵀ + Q
    Trả về (mu_b, Sig_b)."""
    raise NotImplementedError("TODO Bài 1: dự đoán")


def cap_nhat(mu_b, Sig_b, z):
    """Bài 2 (cập nhật): trộn dự đoán với số đo z (một số, vị trí đo được).

        S = C·Σ̄·Cᵀ + R           K = Σ̄·Cᵀ·S⁻¹            (K hình 2×1)
        μ = μ̄ + K·(z − C·μ̄)
        Σ = (I − K·C)·Σ̄·(I − K·C)ᵀ + K·R·Kᵀ                (dạng Joseph, giữ Σ đối xứng, dương)
    Trả về (mu, Sig, K)."""
    raise NotImplementedError("TODO Bài 2: cập nhật")


def loc_kalman(ds_u, ds_z, mu0, Sig0):
    """Bài 3: chạy bộ lọc qua cả chuỗi. Mỗi bước k: du_doan với ds_u[k], rồi cap_nhat với ds_z[k].

    Trả về list các μ (mỗi bước một mảng 2 số: vị trí, vận tốc) sau cập nhật."""
    raise NotImplementedError("TODO Bài 3: vòng lặp Kalman")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong Bài 1–2, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Khai sai độ nhiễu cảm biến R cho bộ lọc (tốt hơn hoặc tệ hơn thật) rồi xem sai RMS.

In [ ]:
import loi_giai
from kiem_tra import DS_U, DS_Z, MU0, SIG0, P, _rms
for sz in [0.05, 0.15, 0.3, 0.6, 1.0]:
    loi_giai.R = np.array([[sz ** 2]])
    p_hat = np.array([m[0] for m in loi_giai.loc_kalman(DS_U, DS_Z, MU0, SIG0)])
    print(f"bộ lọc tưởng σ_z = {sz:4.2f} m (thật 0,3): sai RMS {_rms(p_hat - P) * 100:5.1f} cm")
loi_giai.R = np.array([[0.3 ** 2]])

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap32).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)